# PHASE 3 — Isolation Forest
## FRAUDX — Détection de fraude bancaire par IA au Togo

**Objectif :** Implémenter un modèle non supervisé de détection d'anomalies comme Niveau 1 de l'architecture conceptuelle.

**Jours 13-14 du plan**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix

sns.set_style('whitegrid')

In [ ]:
# === CHARGEMENT des données prétraitées ===
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/FRAUDX/data/'

X_train = pd.read_pickle(f'{data_path}X_train.pkl')
X_test = pd.read_pickle(f'{data_path}X_test.pkl')
y_train = pd.read_pickle(f'{data_path}y_train.pkl')
y_test = pd.read_pickle(f'{data_path}y_test.pkl')

# Conversion en array si nécessaire
if isinstance(y_train, pd.DataFrame):
    y_train = y_train.values.ravel()
if isinstance(y_test, pd.DataFrame):
    y_test = y_test.values.ravel()

print(f'X_train : {X_train.shape}')
print(f'X_test : {X_test.shape}')

---
## JOUR 13 — Implémentation

**Objectif :** Entraîner Isolation Forest sur X_train (sans labels) et scorer X_test.

In [ ]:
# === Isolation Forest avec contamination=0.035 (taux de fraude connu) ===
iso_forest = IsolationForest(
    contamination=0.035,
    random_state=42,
    n_estimators=100,
    max_samples='auto',
    n_jobs=-1
)

# Entraînement (sans utiliser y_train — non supervisé)
iso_forest.fit(X_train)
print('✅ Isolation Forest entraîné sur X_train')

In [ ]:
# === Scores d'anomalie ===
# score_samples : plus le score est négatif, plus la transaction est anormale
scores_train = iso_forest.score_samples(X_train)
scores_test = iso_forest.score_samples(X_test)

# Prédictions : -1 = anomalie (fraude), 1 = normal
preds_train = iso_forest.predict(X_train)
preds_test = iso_forest.predict(X_test)

# Convertir -1/1 en 1/0 pour compatibilité
preds_test_binary = np.where(preds_test == -1, 1, 0)

print(f'Scores train : min={scores_train.min():.4f}, max={scores_train.max():.4f}')
print(f'Scores test : min={scores_test.min():.4f}, max={scores_test.max():.4f}')
print(f'\nAnomalies détectées (test) : {preds_test_binary.sum()} / {len(preds_test_binary)} ({preds_test_binary.mean()*100:.2f}%)')

In [ ]:
# === Alternative : contamination='auto' (discussion mémoire) ===
iso_auto = IsolationForest(
    contamination='auto',
    random_state=42,
    n_estimators=100,
    n_jobs=-1
)
iso_auto.fit(X_train)
preds_auto = iso_auto.predict(X_test)
preds_auto_binary = np.where(preds_auto == -1, 1, 0)

print(f'Avec contamination="auto" : {preds_auto_binary.sum()} anomalies détectées')
print(f'  Soit {preds_auto_binary.mean()*100:.2f}% des transactions')
print(f'\nDiscussion mémoire : contamination="auto" estime automatiquement le taux')
print(f'  d\'anomalies, mais un non-supervisé ne devrait pas "connaître" le taux de fraude.')

---
## JOUR 14 — Évaluation

**Objectif :** Mesurer les performances et analyser les scores.

In [ ]:
# === Métriques ===
recall = recall_score(y_test, preds_test_binary)
precision = precision_score(y_test, preds_test_binary)
f1 = f1_score(y_test, preds_test_binary)

print('=== Isolation Forest — Performances ===')
print(f'Recall (fraudes détectées) : {recall*100:.2f}%')
print(f'Precision : {precision*100:.2f}%')
print(f'F1-Score : {f1*100:.2f}%')
print(f'\nRappel : l\'Isolation Forest est un filtre non supervisé (Niveau 1).')
print(f'Ses performances sont attendues comme inférieures à un supervisé (XGBoost).')

In [ ]:
# === Figure — Boxplot des scores selon isFraud ===
fig, ax = plt.subplots(figsize=(8, 5))

scores_fraud = scores_test[y_test == 1]
scores_normal = scores_test[y_test == 0]

bp = ax.boxplot([scores_normal, scores_fraud], labels=['Non fraude', 'Fraude'],
                widths=0.4, patch_artist=True)
bp['boxes'][0].set_facecolor('steelblue')
bp['boxes'][1].set_facecolor('crimson')

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Score d\'anomalie (score_samples)')
ax.set_title('Figure 3.x — Distribution des scores Isolation Forest par classe', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Interprétation : les transactions frauduleuses devraient avoir des scores')
print('plus négatifs (anomalies) que les transactions normales.')

In [ ]:
# === Recall dans le top 5% des anomalies ===
threshold_top5 = np.percentile(scores_test, 5)
top5_anomalies = scores_test <= threshold_top5
recall_top5 = recall_score(y_test, top5_anomalies)

print(f'Seuil top 5% : {threshold_top5:.4f}')
print(f'Transactions dans le top 5% des anomalies : {top5_anomalies.sum()}')
print(f'Recall dans le top 5% : {recall_top5*100:.2f}%')
print(f'\nIdéalement, le recall dans le top 5% devrait être >50%')
print(f'pour considérer le filtre comme utile.')

In [ ]:
# === Sauvegarde du modèle ===
joblib.dump(iso_forest, 'models/isolation_forest.pkl')
print('✅ Modèle Isolation Forest sauvegardé dans models/isolation_forest.pkl')

---
## Synthèse pour le mémoire (section 3.3)

### Rôle dans l'architecture
L'Isolation Forest constitue le **Niveau 1** de l'architecture à 3 niveaux :
- **Niveau 1** → Isolation Forest : filtre rapide non supervisé
- **Niveau 2** → XGBoost : classification fine supervisée
- **Niveau 3** → LSTM (optionnel) : patterns temporels

### Conclusion attendue
L'Isolation Forest est conceptuellement utile comme premier filtre, mais ses performances
sont inférieures à celles d'un modèle supervisé. Il est conservé dans l'architecture
comme niveau de pré-filtrage pour réduire le volume de transactions à analyser par XGBoost.